# AI-RAG · Week 13 — Notebook 2 (Day 2)
## The Three Tools, Build & Measure

Deeper on each of the three tools — an external API (`search_web`), structured data with **SQL safety rails** (`query_sql`), and compute (`calculate`) — then measure **tool-selection accuracy**, the new KPI #5. Runs **fully offline**.

> Code is verbatim from the package (notebook glue only). The prerequisites re-establish the tools + agent from Notebook 1 so this notebook is self-contained.

### Setup

In [1]:
# --- notebook setup (delivery glue only) -------------------------------------
# 1) ensure pydantic  2) write the offline data files the tools read.
# The agent runs offline by default (deterministic StubRouter + fixtures) — no
# OpenAI/Tavily key needed. The SQLite side-DB is built in the next cell.
try:
    import pydantic  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pydantic"])

from pathlib import Path
Path("data").mkdir(exist_ok=True)
Path("data/web_fixtures.json").write_text(r"""{
  "ai agents": [
    {"title": "State of AI Agents 2026", "snippet": "Agentic systems moved from demos to production in 2026, with tool-use reliability the key differentiator."},
    {"title": "Agent frameworks roundup", "snippet": "Raw loops and graph-based orchestration remain the two dominant patterns."}
  ],
  "openai api": [
    {"title": "OpenAI API pricing", "snippet": "gpt-4o-mini is positioned as the low-cost tier for high-volume workloads."}
  ],
  "cricket": [
    {"title": "Cricket World Cup — result", "snippet": "The most recent tournament concluded with a closely contested final."}
  ],
  "python libraries": [
    {"title": "Trending Python libraries", "snippet": "Data, LLM-tooling, and async libraries dominate this year's trending lists."}
  ],
  "announce": [
    {"title": "OpenAI announcements", "snippet": "Recent announcements focus on tool use, structured outputs, and cost reductions."}
  ],
  "_default": [
    {"title": "Web result", "snippet": "A relevant public-web snippet would appear here (offline fixture)."}
  ]
}""" + "\n")
Path("data/tool_selection_set.jsonl").write_text(r"""{"id": "calc_1", "question": "What is 1350 * 0.18?", "expected_tool": "calculate"}
{"id": "calc_2", "question": "Calculate 4200 / 12.", "expected_tool": "calculate"}
{"id": "calc_3", "question": "How much is 15% of 890?", "expected_tool": "calculate"}
{"id": "calc_4", "question": "What is (320 + 480) * 2?", "expected_tool": "calculate"}
{"id": "calc_5", "question": "What is 900 - 275?", "expected_tool": "calculate"}
{"id": "sql_1", "question": "How many documents are in the knowledge base?", "expected_tool": "query_sql"}
{"id": "sql_2", "question": "How many HR documents do we have?", "expected_tool": "query_sql"}
{"id": "sql_3", "question": "Which department has the most documents?", "expected_tool": "query_sql"}
{"id": "sql_4", "question": "What version is the Acceptable Use Policy?", "expected_tool": "query_sql"}
{"id": "sql_5", "question": "List the Finance documents and when they were last updated.", "expected_tool": "query_sql"}
{"id": "web_1", "question": "What is the latest news on AI agents in 2026?", "expected_tool": "search_web"}
{"id": "web_2", "question": "What is the current price of the OpenAI API for gpt-4o-mini?", "expected_tool": "search_web"}
{"id": "web_3", "question": "Who won the most recent Cricket World Cup?", "expected_tool": "search_web"}
{"id": "web_4", "question": "What are the trending Python libraries this year?", "expected_tool": "search_web"}
{"id": "web_5", "question": "What did OpenAI announce today?", "expected_tool": "search_web"}""" + "\n")
print("wrote offline data files (web fixtures + tool-selection set)")


wrote offline data files (web fixtures + tool-selection set)


### Setup · the side-database

In [2]:
#!/usr/bin/env python3
"""scripts/seed_db.py — (re)create the capstone side database.

W13 gives the assistant structured data to query as a tool. This builds a small
SQLite database with two tables:

  document_metadata  — one row per document in the knowledge base (the structured
                       side-table the SQL tool queries)
  audit_log          — every SQL query the agent runs is logged here (the audit
                       rail for SQL-as-a-tool)

SQLite is local and offline, so the whole SQL-tool exercise runs with no network
and no credentials. Run this once (smoke_test.sh does it for you):

    python3 scripts/seed_db.py
"""
from __future__ import annotations

import sqlite3
from pathlib import Path

DB_PATH = Path("data/capstone_side.db")   # notebook: local file

# A small, realistic slice of the Enterprise Knowledge Assistant's corpus,
# across the three departments the capstone covers.
DOCS = [
    # source, title, doc_type, department, version, updated_at, chunk_count
    ("hr-001", "Leave & Time-Off Policy",          "policy",   "HR",      "v3", "2026-01-14", 42),
    ("hr-002", "Code of Conduct",                  "policy",   "HR",      "v2", "2025-11-02", 55),
    ("hr-003", "Remote Work Guidelines",           "guideline","HR",      "v4", "2026-02-20", 28),
    ("hr-004", "Onboarding Handbook",              "handbook", "HR",      "v1", "2025-06-30", 90),
    ("it-001", "Acceptable Use Policy",            "policy",   "IT",      "v5", "2026-01-30", 37),
    ("it-002", "Security Incident Runbook",        "runbook",  "IT",      "v2", "2026-02-11", 33),
    ("it-003", "VPN & Access Setup",               "guideline","IT",      "v3", "2025-12-05", 19),
    ("it-004", "Data Classification Standard",     "standard", "IT",      "v2", "2026-01-08", 24),
    ("fin-001","Expense Reimbursement Policy",     "policy",   "Finance", "v3", "2026-01-22", 31),
    ("fin-002","Procurement Guidelines",           "guideline","Finance", "v1", "2025-09-15", 40),
    ("fin-003","Travel Policy",                    "policy",   "Finance", "v2", "2025-10-19", 26),
    ("fin-004","Corporate Card Terms",             "terms",    "Finance", "v4", "2026-02-01", 18),
]


def seed(db_path: Path = DB_PATH) -> Path:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    if db_path.exists():
        db_path.unlink()
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    cur.execute("""
        CREATE TABLE document_metadata (
            id           INTEGER PRIMARY KEY,
            source       TEXT NOT NULL,
            title        TEXT NOT NULL,
            doc_type     TEXT NOT NULL,
            department   TEXT NOT NULL,
            version      TEXT NOT NULL,
            updated_at   TEXT NOT NULL,
            chunk_count  INTEGER NOT NULL
        )""")
    cur.executemany(
        "INSERT INTO document_metadata "
        "(source,title,doc_type,department,version,updated_at,chunk_count) "
        "VALUES (?,?,?,?,?,?,?)", DOCS)
    cur.execute("""
        CREATE TABLE audit_log (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            ts         TEXT NOT NULL,
            actor      TEXT NOT NULL,
            request    TEXT,
            sql        TEXT,
            decision   TEXT NOT NULL,
            row_count  INTEGER,
            error      TEXT
        )""")
    con.commit()
    con.close()
    return db_path

seed()
print('side-DB built:', DB_PATH, '· tables: document_metadata, audit_log')


side-DB built: data/capstone_side.db · tables: document_metadata, audit_log


### Prerequisites from Notebook 1

The tool interface, the three tools, the router, and the agent — re-established here so this notebook runs on its own. The new material starts at section 1.

In [3]:
"""src/agents/tools/base.py — the Tool contract.

A tool an LLM can call reliably is four things done well (the W13 principles):
  1. a good NAME       — a verb, specific, snake_case
  2. a typed SCHEMA    — a Pydantic model: required fields, sensible defaults
  3. a DESCRIPTION     — written for the *model*, saying WHEN to use the tool
  4. IDEMPOTENCY        — declared, so the loop knows if a retry is safe

Every tool subclasses `Tool`, sets those four, and implements `run`. The same
definition exports to both OpenAI function-calling and Anthropic tool-use schemas.
"""
from __future__ import annotations

from typing import Any

from pydantic import BaseModel


class Tool:
    name: str = ""
    description: str = ""
    Args: type[BaseModel] = BaseModel
    idempotent: bool = True          # is calling this twice with the same args safe?
    has_side_effects: bool = False   # does it change external state?

    def run(self, **kwargs: Any) -> str:
        raise NotImplementedError

    # ── validation + execution ────────────────────────────────────────────
    def call(self, arguments: dict[str, Any]) -> str:
        """Validate arguments against the schema, then run."""
        args = self.Args(**(arguments or {}))
        return self.run(**args.model_dump())

    # ── provider schema exports ───────────────────────────────────────────
    def parameters_schema(self) -> dict:
        return self.Args.model_json_schema()

    def openai_schema(self) -> dict:
        return {"type": "function",
                "function": {"name": self.name,
                             "description": self.description,
                             "parameters": self.parameters_schema()}}

    def anthropic_schema(self) -> dict:
        return {"name": self.name,
                "description": self.description,
                "input_schema": self.parameters_schema()}


In [4]:
"""src/agents/tools/calculate.py — a computation tool.

Deterministic, offline, and safe: it evaluates an arithmetic expression with a
small AST walker (no `eval`), so an LLM can hand it a calculation and get a
reliable number back. Idempotent and side-effect-free — the easy end of the
tool spectrum, and a good first example of a well-formed tool.
"""
from __future__ import annotations

import ast
import operator

from pydantic import BaseModel, Field

# Tool is defined in the cell above

_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Mod: operator.mod, ast.Pow: operator.pow,
    ast.USub: operator.neg, ast.UAdd: operator.pos, ast.FloorDiv: operator.floordiv,
}


def _eval(node: ast.AST) -> float:
    if isinstance(node, ast.Expression):
        return _eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval(node.operand))
    raise ValueError("unsupported expression — only + - * / % ** ( ) and numbers")


class CalculateArgs(BaseModel):
    expression: str = Field(
        ..., description="An arithmetic expression, e.g. '1350 * 0.18' or '(42 + 8) / 2'.")


class CalculateTool(Tool):
    name = "calculate"
    description = ("Evaluate an arithmetic expression and return the numeric result. "
                   "Use this whenever a question needs exact math — sums, differences, "
                   "products, division, percentages, or powers. Do NOT use it to look up "
                   "facts; it only does arithmetic on numbers you provide.")
    Args = CalculateArgs
    idempotent = True
    has_side_effects = False

    def run(self, expression: str) -> str:
        try:
            result = _eval(ast.parse(expression, mode="eval"))
        except Exception as e:  # noqa: BLE001 - report cleanly to the agent
            return f"calculate error: {e}"
        if isinstance(result, float) and result.is_integer():
            result = int(result)
        return f"{expression} = {result}"


In [5]:
"""src/agents/tools/query_sql.py — SQL as a tool, safely.

Letting an LLM run SQL is powerful and dangerous. This tool ships the three rails
every SQL-as-a-tool needs in production:

  1. AUDIT LOGGING     — every query (allowed or denied) is written to audit_log
  2. READ-ONLY DEFAULT  — only a single SELECT / WITH…SELECT is permitted
  3. CONSTRAINED UPDATES — writes/DDL (INSERT/UPDATE/DELETE/DROP/…) are denied
                          outright (write access is a W18 topic, gated behind
                          injection defenses and a human-approval step)

Everything runs against a local SQLite database, so it works offline with no
credentials. The migration path to Postgres is a config change, not a rewrite
(see docs/runbooks/sql_as_a_tool_safety.md).
"""
from __future__ import annotations

import re
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

from pydantic import BaseModel, Field

# Tool is defined in the cell above

DB_PATH = Path("data/capstone_side.db")   # notebook: local file (built in setup)

# Statements that change data or schema — denied by the read-only rail.
_FORBIDDEN = {"insert", "update", "delete", "drop", "alter", "create", "replace",
              "truncate", "attach", "detach", "pragma", "vacuum", "reindex",
              "grant", "revoke", "begin", "commit"}
_MAX_ROWS = 50


def _strip_comments(sql: str) -> str:
    sql = re.sub(r"--[^\n]*", " ", sql)
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.S)
    return sql.strip()


def is_read_only(sql: str) -> tuple[bool, str]:
    """Return (ok, reason). Only a single SELECT/WITH…SELECT is allowed."""
    clean = _strip_comments(sql)
    if not clean:
        return False, "empty query"
    statements = [s for s in clean.split(";") if s.strip()]
    if len(statements) > 1:
        return False, "multiple statements are not allowed"
    first = statements[0].lstrip().split(None, 1)[0].lower()
    if first not in ("select", "with"):
        return False, f"only SELECT queries are allowed (got '{first.upper()}')"
    words = set(re.findall(r"[a-z_]+", clean.lower()))
    hit = words & _FORBIDDEN
    if hit:
        return False, f"write/DDL keyword not permitted: {', '.join(sorted(hit))}"
    return True, "ok"


class QuerySqlArgs(BaseModel):
    sql: str = Field(..., description="A single read-only SQL SELECT query over the "
                                      "document_metadata table.")
    purpose: str = Field("", description="One phrase describing what the query is for "
                                         "(recorded in the audit log).")


class QuerySqlTool(Tool):
    name = "query_sql"
    description = (
        "Run a read-only SQL SELECT against the knowledge-base metadata database and "
        "return the rows. Use this for questions about the DOCUMENTS THEMSELVES — how "
        "many documents exist, which department owns them, their type, version, "
        "chunk_count, or when they were last updated. Table: document_metadata"
        "(id, source, title, doc_type, department, version, updated_at, chunk_count). "
        "Do NOT use it to answer questions about document CONTENTS — only their metadata.")
    Args = QuerySqlArgs
    idempotent = True            # SELECTs don't change state
    has_side_effects = False

    def __init__(self, db_path: Path | str = DB_PATH):
        self.db_path = Path(db_path)

    def _audit(self, request: str, sql: str, decision: str,
               row_count: int | None, error: str | None) -> None:
        con = sqlite3.connect(self.db_path)
        con.execute(
            "INSERT INTO audit_log (ts, actor, request, sql, decision, row_count, error) "
            "VALUES (?,?,?,?,?,?,?)",
            (datetime.now(timezone.utc).isoformat(timespec="seconds"), "agent",
             request, sql, decision, row_count, error))
        con.commit()
        con.close()

    def run(self, sql: str, purpose: str = "") -> str:
        ok, reason = is_read_only(sql)
        if not ok:
            self._audit(purpose, sql, "denied", None, reason)
            return (f"query_sql DENIED: {reason}. This tool is read-only; write access "
                    f"is not available. (Logged to the audit trail.)")
        try:
            con = sqlite3.connect(self.db_path)
            cur = con.execute(sql)
            rows = cur.fetchmany(_MAX_ROWS)
            cols = [d[0] for d in cur.description] if cur.description else []
            con.close()
        except Exception as e:  # noqa: BLE001
            self._audit(purpose, sql, "error", None, str(e))
            return f"query_sql error: {e}"
        self._audit(purpose, sql, "allowed", len(rows), None)
        if not rows:
            return "query_sql: (0 rows)"
        header = " | ".join(cols)
        body = "\n".join(" | ".join(str(v) for v in r) for r in rows)
        more = "" if len(rows) < _MAX_ROWS else f"\n… (capped at {_MAX_ROWS} rows)"
        return f"{header}\n{'-' * len(header)}\n{body}{more}"


In [6]:
"""src/agents/tools/search_web.py — an external-API tool (web search).

Wraps a web-search API (Tavily) so the agent can reach information beyond the
knowledge base — current events, prices, anything not in the corpus. This is the
"API-calling agent" pattern: the LLM decides it needs fresh external info and
calls out.

Web search has real cost and latency, so the description tells the model to use
it only when the answer isn't in the corpus. For offline runs and tests, the tool
falls back to a local fixture file, so the whole thing works with no network and
no TAVILY_API_KEY.
"""
from __future__ import annotations

import json
import os
from pathlib import Path

from pydantic import BaseModel, Field

# Tool is defined in the cell above

FIXTURES_PATH = Path("data/web_fixtures.json")   # notebook: local file (written in setup)


class SearchWebArgs(BaseModel):
    query: str = Field(..., description="A concise web-search query.")
    max_results: int = Field(3, ge=1, le=5, description="How many results to return.")


class SearchWebTool(Tool):
    name = "search_web"
    description = (
        "Search the public web and return short result snippets. Use this ONLY when the "
        "answer is not in the internal knowledge base — for current events, live figures, "
        "recent releases, or anything outside company documents. It has cost and latency, "
        "so prefer the internal tools when they can answer. Returns titles and snippets, "
        "not full pages.")
    Args = SearchWebArgs
    idempotent = True            # reading search results doesn't change state
    has_side_effects = False

    def __init__(self, offline: bool | None = None):
        # offline when explicitly set, or when there is no API key available
        self.offline = (not os.environ.get("TAVILY_API_KEY")) if offline is None else offline

    def _fixture(self, query: str, k: int) -> str:
        try:
            data = json.loads(FIXTURES_PATH.read_text())
        except Exception:
            data = {"_default": [{"title": "No fixture", "snippet": "offline stub result."}]}
        results = None
        for key, res in data.items():
            if key != "_default" and key.lower() in query.lower():
                results = res
                break
        results = (results or data.get("_default", []))[:k]
        lines = [f"[web:{query}] (offline fixture)"]
        for r in results:
            lines.append(f"- {r['title']}: {r['snippet']}")
        return "\n".join(lines)

    def run(self, query: str, max_results: int = 3) -> str:
        if self.offline:
            return self._fixture(query, max_results)
        # live path (needs network + TAVILY_API_KEY)
        try:
            from tavily import TavilyClient
            client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
            resp = client.search(query=query, max_results=max_results)
            lines = [f"[web:{query}]"]
            for r in resp.get("results", [])[:max_results]:
                lines.append(f"- {r.get('title','')}: {r.get('content','')[:160]}")
            return "\n".join(lines)
        except Exception as e:  # noqa: BLE001
            return f"search_web error: {e} (falling back offline)\n" + self._fixture(query, max_results)


In [7]:
"""src/agents/router.py — deciding which tool to call.

The router is the "brain" of the agent: given a question and the available tools,
it returns either a TOOL CALL (name + arguments) or a FINAL ANSWER.

Two implementations:

  LLMRouter  — the real thing: OpenAI function calling picks the tool. Needs
               OPENAI_API_KEY + network.
  StubRouter — a deterministic, offline stand-in that routes by keyword. It exists
               so the agent loop, the tool-selection harness, and the tests all run
               with no API key. It is NOT how you'd route in production — it's a
               test double for the *mechanics*.

A router returns a Decision: either .tool_call = (name, args) or .final = "answer".
"""
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass

# Tool is defined in the cell above


@dataclass
class Decision:
    tool_call: tuple[str, dict] | None = None
    final: str | None = None


# ─── deterministic offline router ───────────────────────────────────────────
class StubRouter:
    """Routes by keyword; one tool call, then a final answer synthesised from the
    observation. Deterministic — same question always routes the same way."""

    def route(self, question: str, tools: list[Tool], history: list[dict]) -> Decision:
        # If a tool has already run this turn, produce the final answer.
        if any(m.get("role") == "tool" for m in history):
            obs = next(m["content"] for m in reversed(history) if m.get("role") == "tool")
            return Decision(final=f"Based on the tool result: {obs}")

        q = question.lower()
        names = {t.name for t in tools}

        # 1) arithmetic → calculate
        if "calculate" in names and re.search(r"\d\s*[-+*/%]\s*\d|percent|%|how much is", q):
            expr = self._extract_expr(question)
            return Decision(tool_call=("calculate", {"expression": expr}))

        # 2) metadata about the documents → query_sql
        sql_signals = ("how many document", "how many doc", "number of document", "which department",
                       "what version", "latest version", "list the", "count", "how many hr",
                       "how many it", "how many finance", "updated", "chunk", "doc_type", "document type")
        if "query_sql" in names and any(s in q for s in sql_signals):
            return Decision(tool_call=("query_sql", {"sql": self._canned_sql(q),
                                                     "purpose": "answer a metadata question"}))

        # 3) external / current info → search_web
        web_signals = ("latest", "current", "today", "news", "price", "release", "who won",
                       "weather", "stock", "recent", "2026", "trending", "announced")
        if "search_web" in names and any(s in q for s in web_signals):
            return Decision(tool_call=("search_web", {"query": question, "max_results": 3}))

        # default fallback: search_web if present, else the first tool
        if "search_web" in names:
            return Decision(tool_call=("search_web", {"query": question, "max_results": 3}))
        first = tools[0]
        return Decision(tool_call=(first.name, {}))

    @staticmethod
    def _extract_expr(question: str) -> str:
        m = re.search(r"[0-9][0-9\.\s\+\-\*/%\(\)]*[0-9\)]", question)
        return (m.group(0).strip() if m else "0")

    @staticmethod
    def _canned_sql(q: str) -> str:
        # A safe SELECT that satisfies the metadata question type (selection harness
        # only checks the tool choice, not the exact SQL).
        for dept in ("hr", "it", "finance"):
            if dept in q:
                return ("SELECT COUNT(*) AS n FROM document_metadata "
                        f"WHERE department = '{dept.upper() if dept != 'it' else 'IT'}'")
        if "version" in q:
            return "SELECT source, title, version FROM document_metadata ORDER BY source"
        return "SELECT COUNT(*) AS total_documents FROM document_metadata"


# ─── real OpenAI function-calling router ────────────────────────────────────
class LLMRouter:
    """Uses OpenAI function calling to choose a tool. Needs OPENAI_API_KEY."""

    def __init__(self, model: str = "gpt-5-mini"):
        from openai import OpenAI
        self.client = OpenAI()
        self.model = model
        self.system = ("You are a helpful assistant with tools. Use a tool when it can "
                       "answer the question; otherwise answer directly. Prefer internal "
                       "tools over web search when they suffice.")

    def route(self, question: str, tools: list[Tool], history: list[dict]) -> Decision:
        messages = [{"role": "system", "content": self.system},
                    {"role": "user", "content": question}, *history]
        resp = self.client.chat.completions.create(
            model=self.model, messages=messages,
            tools=[t.openai_schema() for t in tools], tool_choice="auto")
        msg = resp.choices[0].message
        if msg.tool_calls:
            call = msg.tool_calls[0]
            args = json.loads(call.function.arguments or "{}")
            return Decision(tool_call=(call.function.name, args))
        return Decision(final=msg.content or "")


In [8]:
"""src/agents/three_tool_agent.py — the agent loop.

The whole of "an agent" is this loop:

    decide (which tool?) → call it → observe the result → decide again → answer

That's it. The intelligence is in the router (which tool) and the tools (what they
do); the loop just runs the cycle until the router produces a final answer or we
hit a step budget. This is the raw loop, built by hand — no framework (LangGraph
arrives in a later week).

The agent is provider-agnostic: give it any router (LLMRouter for real OpenAI
function calling, or StubRouter for deterministic offline runs) and the three
tools.
"""
from __future__ import annotations

import json
from typing import Any

# Decision, StubRouter, Tool, and the three Tool classes are defined in the cells above


def default_tools(offline: bool = True) -> list[Tool]:
    """The three W13 tools: one external API, one over structured data, one compute."""
    return [SearchWebTool(offline=offline), QuerySqlTool(), CalculateTool()]


class Agent:
    def __init__(self, tools: list[Tool], router: Any):
        self.tools = tools
        self.by_name = {t.name: t for t in tools}
        self.router = router

    def run(self, question: str, max_steps: int = 4) -> dict:
        history: list[dict] = []          # OpenAI-style assistant/tool turns
        steps: list[dict] = []
        for i in range(max_steps):
            decision: Decision = self.router.route(question, self.tools, history)
            if decision.final is not None:
                return {"question": question, "answer": decision.final, "steps": steps,
                        "tool_used": steps[-1]["tool"] if steps else None, "stopped": False}
            name, args = decision.tool_call
            tool = self.by_name.get(name)
            if tool is None:
                obs = f"error: unknown tool '{name}'"
            else:
                try:
                    obs = tool.call(args)
                except Exception as e:  # noqa: BLE001 - surface to the agent, don't crash
                    obs = f"{name} argument error: {e}"
            steps.append({"tool": name, "args": args, "observation": obs})
            call_id = f"call_{i}"
            history.append({"role": "assistant", "content": None,
                            "tool_calls": [{"id": call_id, "type": "function",
                                            "function": {"name": name,
                                                         "arguments": json.dumps(args)}}]})
            history.append({"role": "tool", "tool_call_id": call_id, "content": obs})
        return {"question": question, "answer": "(stopped: max steps reached)",
                "steps": steps, "tool_used": steps[-1]["tool"] if steps else None, "stopped": True}


def build_agent(offline: bool = True, use_llm: bool = False) -> Agent:
    """Construct the 3-tool agent. Default: fully offline with the StubRouter."""
    tools = default_tools(offline=offline)
    if use_llm:
        pass  # LLMRouter is defined above
        return Agent(tools, LLMRouter())
    return Agent(tools, StubRouter())


## 1 · Tool 1 — `search_web`: an API-calling tool with judgment

Web search reaches outside the corpus for current or external information — but it has **cost and latency**, so its description tells the model to prefer the internal tools when they can answer. Offline it returns fixtures; the live path uses Tavily. Run it:

In [9]:
web = SearchWebTool(offline=True)
print(web.call({'query': 'latest news on AI agents in 2026', 'max_results': 2}))

[web:latest news on AI agents in 2026] (offline fixture)
- State of AI Agents 2026: Agentic systems moved from demos to production in 2026, with tool-use reliability the key differentiator.
- Agent frameworks roundup: Raw loops and graph-based orchestration remain the two dominant patterns.


## 2 · Tool 2 — `query_sql`: the SQL safety rails, in depth

Letting an LLM run SQL is powerful and dangerous. Three rails, shown live: **audit logging** (every query, allowed or denied), **read-only default** (only a single `SELECT`), and **denied writes/DDL**. A `SELECT` is allowed; a `DELETE` is refused and logged. *(Mind PII & retention in what the audit log stores — NL requests can carry sensitive data; W28 covers compliance.)*

In [10]:
sql = QuerySqlTool()
print('① SELECT (allowed):')
print(sql.call({'sql': 'SELECT department, COUNT(*) AS n FROM document_metadata GROUP BY department',
                'purpose': 'count documents by department'}))
print('\n② Write (DENIED + logged):')
print(sql.call({'sql': "DELETE FROM document_metadata WHERE department = 'HR'",
                'purpose': 'attempt a delete'}))
print('\n③ The audit trail — both attempts recorded:')
import sqlite3
con = sqlite3.connect('data/capstone_side.db')
for row in con.execute('SELECT decision, row_count, substr(sql,1,42) AS sql FROM audit_log ORDER BY id'):
    print('  ', row)
con.close()

① SELECT (allowed):
department | n
--------------
Finance | 4
HR | 4
IT | 4

② Write (DENIED + logged):
query_sql DENIED: only SELECT queries are allowed (got 'DELETE'). This tool is read-only; write access is not available. (Logged to the audit trail.)

③ The audit trail — both attempts recorded:
   ('allowed', 3, 'SELECT department, COUNT(*) AS n FROM docu')
   ('denied', None, 'DELETE FROM document_metadata WHERE depart')


## 3 · Tool 3 — `calculate`: the clean example

Quick — the safe, well-designed tool that embodies all four principles (already inlined above). It evaluates arithmetic with no `eval`:

In [11]:
print('(320 + 480) * 2 →', CalculateTool().call({'expression': '(320 + 480) * 2'}))

(320 + 480) * 2 → (320 + 480) * 2 = 1600


## 4 · Measure — tool-selection accuracy (`scripts/selection_eval.py`)

The new **KPI #5**. For each labelled question, run the agent and check whether it picked the **right tool**. Reports overall accuracy + a per-tool breakdown. Aim ≥ 80% — and if you're below that with the real model, the fix is almost always in your tool **descriptions**, not the loop.

In [12]:
#!/usr/bin/env python3
"""scripts/selection_eval.py — tool-selection accuracy.

The W13 KPI. For each labelled question, run the agent and check whether it picked
the RIGHT tool for the job. Reports overall accuracy and a per-tool breakdown.

    python3 scripts/selection_eval.py            # offline (StubRouter), no keys needed
    python3 scripts/selection_eval.py --llm      # real OpenAI function calling (needs a key)

Aim for >= 80%. If you're below that with the real model, the fix is almost always
in your tool DESCRIPTIONS, not the loop — see docs/tool_design_principles.md.
"""
from __future__ import annotations

import json
import sys
from pathlib import Path


# build_agent is defined in the cells above

DATA = Path("data/tool_selection_set.jsonl")   # notebook: local file (written in setup)


def load_set(path: Path = DATA) -> list[dict]:
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()]


def evaluate_selection(agent, questions: list[dict]) -> dict:
    rows, correct = [], 0
    per_tool: dict[str, list[int]] = {}
    for q in questions:
        result = agent.run(q["question"])
        chosen = result["steps"][0]["tool"] if result["steps"] else "(none)"
        ok = chosen == q["expected_tool"]
        correct += int(ok)
        per_tool.setdefault(q["expected_tool"], [0, 0])
        per_tool[q["expected_tool"]][0] += int(ok)
        per_tool[q["expected_tool"]][1] += 1
        rows.append({"id": q["id"], "expected": q["expected_tool"], "chosen": chosen, "ok": ok})
    n = len(questions)
    return {"rows": rows, "n": n, "correct": correct,
            "accuracy": round(correct / n, 3) if n else None,
            "per_tool": {k: round(v[0] / v[1], 3) for k, v in per_tool.items()}}


def format_table(result: dict) -> str:
    lines = ["", "  id        expected     chosen       ok", "  " + "-" * 44]
    for r in result["rows"]:
        lines.append(f"  {r['id']:<9} {r['expected']:<12} {r['chosen']:<12} "
                     f"{'✓' if r['ok'] else '✗'}")
    lines += ["  " + "-" * 44, "",
              f"  Tool-selection accuracy: {result['accuracy']}  "
              f"({result['correct']}/{result['n']})"]
    for tool, acc in result["per_tool"].items():
        lines.append(f"    {tool:<12} {acc}")
    lines.append("")
    return "\n".join(lines)


**Run the selection harness** over the 15-question labelled set:

In [13]:
agent = build_agent(offline=True)
result = evaluate_selection(agent, load_set())
print(format_table(result))


  id        expected     chosen       ok
  --------------------------------------------
  calc_1    calculate    calculate    ✓
  calc_2    calculate    calculate    ✓
  calc_3    calculate    calculate    ✓
  calc_4    calculate    calculate    ✓
  calc_5    calculate    calculate    ✓
  sql_1     query_sql    query_sql    ✓
  sql_2     query_sql    query_sql    ✓
  sql_3     query_sql    query_sql    ✓
  sql_4     query_sql    query_sql    ✓
  sql_5     query_sql    query_sql    ✓
  web_1     search_web   search_web   ✓
  web_2     search_web   search_web   ✓
  web_3     search_web   search_web   ✓
  web_4     search_web   search_web   ✓
  web_5     search_web   search_web   ✓
  --------------------------------------------

  Tool-selection accuracy: 1.0  (15/15)
    calculate    1.0
    query_sql    1.0
    search_web   1.0



## Wrap

You can now expose Python functions as tools for both providers from one definition, build an API-calling + SQL agent with real safety rails, and **measure tool-selection accuracy** — KPI #5 (tool success) is now live. The tool decisions are recorded in ADR §7. Next: **W14** — the raw agent loop and failure-oriented design.

*These notebooks are for **seeing and explaining** the agent; in the repo the same code is a pytest-tested package with a `smoke_test.sh` — the notebook to understand it, the package to run it.*